In [11]:
import io
import os
from dotenv import load_dotenv

import fitz  # PyMuPDF
from google.cloud import vision
from google import genai
from google.genai import types

import re
import json
import pdfplumber
from pathlib import Path

from pydantic import BaseModel
from typing import Optional, List

In [12]:
# defining output models
class ReceiptItem(BaseModel):
    name: str
    quantity: float
    unit_price: float
    total_price: float


class ReceiptData(BaseModel):
    store: str
    order_number: Optional[str]
    date: Optional[str]
    currency: str
    items: List[ReceiptItem]
    summary: ReceiptSummary


class ReceiptSummary(BaseModel):
    number_of_items: int
    subtotal: float
    discount: float
    delivery_fee: float
    service_fee: float
    tax: float
    tip: float
    total: float

In [13]:
load_dotenv()
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "ocr_demo_key.json"  # generate this from Google OCR GCP service
)
WORD = re.compile(r"\w+")

In [14]:
api_key = os.getenv("GEMINI_API_KEY")  # generate gemini api key
client = genai.Client(api_key=api_key)

In [15]:
def detect_text(path):
    """
    Detects text in a file using Google Cloud Vision OCR.
    Handles images and multi-page PDFs by converting PDF pages to images.
    """
    client = vision.ImageAnnotatorClient()
    file_ext = Path(path).suffix.lower()
    all_text = []

    image_contents = []

    if file_ext == ".pdf":
        # opening PDF and iterating through all pages
        pdf_document = fitz.open(path)
        for page_num in range(len(pdf_document)):
            page = pdf_document[page_num]

            # convert each page to an image
            matrix = fitz.Matrix(2, 2)
            pix = page.get_pixmap(matrix=matrix)
            image_contents.append(pix.tobytes("png"))
        pdf_document.close()
    else:
        # Handle standard image files (png, jpg, etc.)
        with open(path, "rb") as image_file:
            image_contents.append(image_file.read())

    # Process each image/page through Vision OCR
    for content in image_contents:
        image = vision.Image(content=content)

        # We use document_text_detection for better handling of dense text/receipts
        response = client.document_text_detection(image=image)

        if response.error.message:
            raise Exception(f"Vision API Error: {response.error.message}")

        # text_annotations[0] contains the entire page's text as a single string
        if response.text_annotations:
            page_text = response.text_annotations[0].description
            all_text.append(page_text)

    return "\n--- Page Break ---\n".join(all_text)

In [16]:
RECEIPT_PARSER_PROMPT = """
Extract receipt data with these rules:

Store Name: Use the parent platform/marketplace (Amazon not "Acme Corp", Uber Eats not "Joe's Pizza", Instacart not "Kroger"). Look for "Sold by", "Fulfilled by", "Marketplace".

Items: Extract all line items. Exclude service fees (Delivery Fee, Service Fee, etc.).

Pricing: Ensure unit_price × quantity = total_price. Handle $, £, €.

Dates: Use Invoice/Transaction Date (not Order Date for Amazon). Format: YYYY-MM-DD.

Default missing numeric fields to 0.
"""

In [17]:
def parse_result(receipt_text):
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=receipt_text,
            config={
                "system_instruction": RECEIPT_PARSER_PROMPT,
                "response_mime_type": "application/json",
                "response_schema": ReceiptData.model_json_schema(),
            },
        )
        return ReceiptData.model_validate_json(response.text)
    except Exception as e:
        print(f"Error type: {type(e).__name__}")
        print(f"Error message: {str(e)}")
        print(f"Full error: {repr(e)}")
        raise

In [ ]:
path = "./1DigiKey.pdf"

In [ ]:
result = detect_text(path)
print(result)

In [ ]:
import time

start = time.perf_counter()

output = parse_result(result)

end = time.perf_counter()
print(f"Parsing took {end - start:.4f} seconds")

In [ ]:
print(output)

In [ ]:
print(output.store)

In [ ]:
print(output.date)

In [ ]:
for item in output.items:
    print(item)

In [ ]:
for item in output.summary:
    print(item)